# DAY 11 - Real-World Project: Retail Sales Analytics

## Project Brief
You are a **Data Analyst** hired by a retail company.
They give you 2 years of sales data and ask:

1. Which products/categories make the most money?
2. Which regions are performing well?
3. Is the business growing?
4. Who are our most valuable customers?
5. What are the seasonal patterns?
6. Where are we losing money (returns)?
7. What should we focus on next quarter?

**Your job: Answer all these questions with data and charts.**

---

## Workflow
```
Step 1: Load & Understand Data
Step 2: Clean Data
Step 3: Feature Engineering
Step 4: Exploratory Data Analysis
Step 5: Business KPIs
Step 6: Time Series Analysis
Step 7: Customer Segmentation
Step 8: Executive Dashboard
Step 9: Recommendations
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 150)
np.random.seed(42)
print('All libraries loaded!')

---
## STEP 1: Generate Realistic Retail Dataset

In [ ]:
# ── Dataset Configuration ──────────────────────────────────────
n = 5000  # 5000 orders

categories = {
    'Electronics':  {'products': ['Laptop','Smartphone','Tablet','Earbuds','Smartwatch'],
                     'price_range': (8000, 80000), 'weight': 0.20},
    'Clothing':     {'products': ['T-Shirt','Jeans','Dress','Jacket','Shoes'],
                     'price_range': (500, 5000),   'weight': 0.22},
    'Books':        {'products': ['Fiction','Non-Fiction','Academic','Comics','Self-Help'],
                     'price_range': (200, 1500),   'weight': 0.12},
    'Home & Garden':{'products': ['Sofa','Lamp','Curtains','Plant Pot','Cookware'],
                     'price_range': (1000, 30000), 'weight': 0.18},
    'Sports':       {'products': ['Yoga Mat','Dumbbells','Cycle','Shoes','Racket'],
                     'price_range': (500, 15000),  'weight': 0.15},
    'Beauty':       {'products': ['Moisturizer','Serum','Lipstick','Shampoo','Perfume'],
                     'price_range': (300, 4000),   'weight': 0.13},
}

regions   = ['North', 'South', 'East', 'West', 'Central']
city_map  = {
    'North': ['Delhi','Chandigarh','Ludhiana'],
    'South': ['Bangalore','Chennai','Hyderabad'],
    'East':  ['Kolkata','Bhubaneswar','Patna'],
    'West':  ['Mumbai','Pune','Ahmedabad'],
    'Central':['Bhopal','Nagpur','Indore']
}
ship_modes = ['Standard','Express','Overnight']
ship_cost  = {'Standard': 50, 'Express': 150, 'Overnight': 300}
cust_segments = ['New', 'Regular', 'Premium', 'VIP']

# ── Generate orders ───────────────────────────────────────────
cat_list  = list(categories.keys())
cat_probs = [categories[c]['weight'] for c in cat_list]

order_categories = np.random.choice(cat_list, n, p=cat_probs)
order_products   = [
    np.random.choice(categories[c]['products']) for c in order_categories
]
order_prices = [
    np.random.randint(*categories[c]['price_range'])
    for c in order_categories
]

# Seasonal order dates (more in Nov-Dec for festive season)
all_dates = pd.date_range('2022-01-01', '2023-12-31', freq='D')
date_weights = np.ones(len(all_dates))
for i, d in enumerate(all_dates):
    if d.month in [10, 11, 12]: date_weights[i] = 2.5  # Festive boost
    elif d.month in [7, 8]:     date_weights[i] = 1.5  # Summer sale
    if d.dayofweek in [5, 6]:   date_weights[i] *= 1.3  # Weekend boost
date_weights /= date_weights.sum()
order_dates = np.random.choice(all_dates, n, p=date_weights)

region_orders = np.random.choice(regions, n, p=[0.28,0.22,0.18,0.22,0.10])
cities  = [np.random.choice(city_map[r]) for r in region_orders]

df = pd.DataFrame({
    'order_id':       range(10001, 10001 + n),
    'order_date':     order_dates,
    'customer_id':    np.random.randint(1001, 2001, n),
    'customer_segment': np.random.choice(cust_segments, n, p=[0.35,0.35,0.20,0.10]),
    'customer_age':   np.random.randint(18, 65, n),
    'customer_gender':np.random.choice(['Male','Female'], n),
    'category':       order_categories,
    'product':        order_products,
    'unit_price':     order_prices,
    'quantity':       np.random.randint(1, 5, n),
    'discount_pct':   np.random.choice([0,5,10,15,20,25,30], n, p=[0.30,0.15,0.20,0.15,0.10,0.07,0.03]),
    'shipping_mode':  np.random.choice(ship_modes, n, p=[0.55,0.30,0.15]),
    'region':         region_orders,
    'city':           cities,
    'is_returned':    np.random.choice([True,False], n, p=[0.07,0.93]),
})

# Computed columns
df['gross_sales']   = df['unit_price'] * df['quantity']
df['discount_amt']  = (df['gross_sales'] * df['discount_pct'] / 100).astype(int)
df['net_sales']     = df['gross_sales'] - df['discount_amt']
df['shipping_cost'] = df['shipping_mode'].map(ship_cost)
df['profit']        = (df['net_sales'] * 0.25 - df['shipping_cost']).astype(int)
df['profit']        = np.where(df['is_returned'], -df['unit_price'] * 0.3, df['profit']).astype(int)
df['order_date']    = pd.to_datetime(df['order_date'])
df['year']          = df['order_date'].dt.year
df['month']         = df['order_date'].dt.month
df['month_name']    = df['order_date'].dt.month_name()
df['quarter']       = df['order_date'].dt.quarter
df['day_of_week']   = df['order_date'].dt.day_name()
df['week']          = df['order_date'].dt.isocalendar().week.astype(int)

print(f'Dataset created: {df.shape[0]:,} orders')
print(df.head(3))

---
## STEP 2: Data Quality Check

In [ ]:
print('='*65)
print('DATA QUALITY REPORT')
print('='*65)
print(f'Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date Range     : {df["order_date"].min().date()} to {df["order_date"].max().date()}')
print(f'Missing Values : {df.isnull().sum().sum()}')
print(f'Duplicates     : {df.duplicated().sum()}')
print()
df.info()

---
## STEP 3: Business KPIs — The Numbers That Matter

In [ ]:
print('='*65)
print('KEY PERFORMANCE INDICATORS (KPIs)')
print('='*65)

total_revenue   = df['net_sales'].sum()
total_profit    = df['profit'].sum()
total_orders    = len(df)
unique_customers= df['customer_id'].nunique()
avg_order_val   = df['net_sales'].mean()
return_rate     = df['is_returned'].mean() * 100
profit_margin   = (total_profit / total_revenue) * 100
repeat_cust     = df.groupby('customer_id').size()
repeat_pct      = (repeat_cust > 1).mean() * 100

kpis = [
    ('Total Revenue',        f'Rs {total_revenue:>15,.0f}'),
    ('Total Profit',         f'Rs {total_profit:>15,.0f}'),
    ('Profit Margin',        f'{profit_margin:>16.1f}%'),
    ('Total Orders',         f'{total_orders:>17,}'),
    ('Unique Customers',     f'{unique_customers:>17,}'),
    ('Avg Order Value',      f'Rs {avg_order_val:>15,.0f}'),
    ('Return Rate',          f'{return_rate:>16.1f}%'),
    ('Repeat Customer Rate', f'{repeat_pct:>16.1f}%'),
]

for name, val in kpis:
    print(f'  {name:<22}: {val}')

---
## STEP 4: Category Performance Analysis

In [ ]:
cat_perf = df.groupby('category').agg(
    Orders     =('order_id',   'count'),
    Revenue    =('net_sales',  'sum'),
    Profit     =('profit',     'sum'),
    Avg_Order  =('net_sales',  'mean'),
    Returns    =('is_returned','sum'),
    Avg_Discount=('discount_pct','mean')
).round(1)

cat_perf['Return_Rate%']   = (cat_perf['Returns'] / cat_perf['Orders'] * 100).round(1)
cat_perf['Profit_Margin%'] = (cat_perf['Profit'] / cat_perf['Revenue'] * 100).round(1)
cat_perf = cat_perf.sort_values('Revenue', ascending=False)

print('Category Performance Summary:')
print(cat_perf[['Orders','Revenue','Profit','Avg_Order','Return_Rate%','Profit_Margin%']])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Category Analysis', fontsize=16, fontweight='bold')

# Revenue
axes[0].barh(cat_perf.index, cat_perf['Revenue'], color=plt.cm.Set2.colors[:6])
axes[0].set_title('Revenue by Category')
axes[0].set_xlabel('Revenue (Rs)')

# Profit Margin
colors_pm = ['green' if v > 0 else 'red' for v in cat_perf['Profit_Margin%']]
axes[1].barh(cat_perf.index, cat_perf['Profit_Margin%'], color=colors_pm)
axes[1].axvline(0, color='black', lw=1)
axes[1].set_title('Profit Margin %')
axes[1].set_xlabel('Margin %')

# Return Rate
axes[2].barh(cat_perf.index, cat_perf['Return_Rate%'], color='tomato', alpha=0.8)
axes[2].set_title('Return Rate %')
axes[2].set_xlabel('Return %')

plt.tight_layout()
plt.show()

---
## STEP 5: Regional Performance

In [ ]:
region_perf = df.groupby('region').agg(
    Orders  =('order_id',  'count'),
    Revenue =('net_sales', 'sum'),
    Profit  =('profit',    'sum'),
    Customers=('customer_id','nunique')
).sort_values('Revenue', ascending=False)

region_perf['Revenue_per_Order']    = (region_perf['Revenue'] / region_perf['Orders']).round(0)
region_perf['Revenue_per_Customer'] = (region_perf['Revenue'] / region_perf['Customers']).round(0)

print('Regional Performance:')
print(region_perf)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Pie
axes[0].pie(region_perf['Revenue'], labels=region_perf.index,
            autopct='%1.1f%%', startangle=90,
            colors=plt.cm.Set1.colors[:5])
axes[0].set_title('Revenue Share by Region', fontsize=14)

# Bar: Revenue per customer
axes[1].bar(region_perf.index, region_perf['Revenue_per_Customer'],
            color=plt.cm.Set1.colors[:5], edgecolor='black')
axes[1].set_title('Revenue per Customer by Region', fontsize=14)
axes[1].set_ylabel('Revenue (Rs)')
for i, (idx, val) in enumerate(region_perf['Revenue_per_Customer'].items()):
    axes[1].text(i, val + 100, f'Rs {val:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## STEP 6: Time Series — Growth Analysis

In [ ]:
monthly = df.groupby(['year', 'month']).agg(
    Revenue=('net_sales', 'sum'),
    Orders =('order_id',  'count'),
    Profit =('profit',    'sum')
).reset_index()
monthly['date'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
monthly = monthly.sort_values('date').reset_index(drop=True)
monthly['revenue_ma3']   = monthly['Revenue'].rolling(3).mean()
monthly['mom_growth_pct']= monthly['Revenue'].pct_change() * 100

fig, axes = plt.subplots(3, 1, figsize=(15, 13))
fig.suptitle('Time Series — Revenue & Growth', fontsize=18, fontweight='bold')

# Revenue bars + trend line
colors_yr = ['steelblue' if y == 2022 else 'tomato' for y in monthly['year']]
axes[0].bar(range(len(monthly)), monthly['Revenue'],
            color=colors_yr, alpha=0.8, edgecolor='white')
axes[0].plot(range(len(monthly)), monthly['revenue_ma3'],
             color='black', lw=2.5, label='3-Month MA')
axes[0].set_xticks(range(len(monthly)))
labels = [f"{row['year']}-{row['month']:02d}" for _, row in monthly.iterrows()]
axes[0].set_xticklabels(labels, rotation=45, fontsize=8)
axes[0].set_title('Monthly Revenue (Blue=2022, Red=2023)')
axes[0].set_ylabel('Revenue (Rs)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Orders
axes[1].plot(range(len(monthly)), monthly['Orders'], 'o-',
             color='steelblue', lw=2, ms=6)
axes[1].set_xticks(range(len(monthly)))
axes[1].set_xticklabels(labels, rotation=45, fontsize=8)
axes[1].set_title('Monthly Order Volume')
axes[1].set_ylabel('Orders')
axes[1].grid(True, alpha=0.3)

# MoM growth
growth = monthly['mom_growth_pct'].dropna()
gcolors = ['green' if v > 0 else 'red' for v in growth]
axes[2].bar(range(1, len(monthly)), growth.values, color=gcolors, alpha=0.8)
axes[2].axhline(0, color='black', lw=1)
axes[2].set_xticks(range(1, len(monthly)))
axes[2].set_xticklabels(labels[1:], rotation=45, fontsize=8)
axes[2].set_title('Month-over-Month Revenue Growth (%)')
axes[2].set_ylabel('Growth %')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# YoY comparison
y2022 = monthly[monthly['year']==2022]['Revenue'].sum()
y2023 = monthly[monthly['year']==2023]['Revenue'].sum()
print(f'2022 Total Revenue: Rs {y2022:>12,.0f}')
print(f'2023 Total Revenue: Rs {y2023:>12,.0f}')
print(f'YoY Growth        : {(y2023/y2022-1)*100:>12.1f}%')

---
## STEP 7: Customer Segmentation (RFM Analysis)

**RFM = Recency, Frequency, Monetary**

This is the most used customer analysis in retail.
- **Recency** → How recently did they buy?
- **Frequency** → How often do they buy?
- **Monetary** → How much do they spend?

In [ ]:
# RFM Calculation
today = pd.Timestamp('2024-01-01')

rfm = df.groupby('customer_id').agg(
    Recency   =('order_date',  lambda x: (today - x.max()).days),
    Frequency =('order_id',    'count'),
    Monetary  =('net_sales',   'sum')
).reset_index()

# Score each dimension 1-4
rfm['R_Score'] = pd.qcut(rfm['Recency'],  4, labels=[4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# Segment customers
def rfm_segment(score):
    if score >= 10: return 'Champions'
    elif score >= 8: return 'Loyal'
    elif score >= 6: return 'Potential'
    elif score >= 4: return 'At Risk'
    else:            return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(rfm_segment)

print('RFM Analysis:')
print(rfm.head(10))
print('\nCustomer Segments:')
print(rfm['Segment'].value_counts())

In [ ]:
seg_summary = rfm.groupby('Segment').agg(
    Customers=('customer_id', 'count'),
    Avg_Recency=('Recency',   'mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary=('Monetary',  'mean')
).round(1).sort_values('Avg_Monetary', ascending=False)

print('Segment Summary:')
print(seg_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Segment size
seg_counts = rfm['Segment'].value_counts()
order_seg = ['Champions','Loyal','Potential','At Risk','Lost']
seg_counts = seg_counts.reindex(order_seg, fill_value=0)
cols = ['gold','steelblue','lightgreen','orange','tomato']
axes[0].bar(seg_counts.index, seg_counts.values, color=cols, edgecolor='black')
for i, v in enumerate(seg_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Customer Count by Segment', fontsize=14)
axes[0].set_ylabel('Number of Customers')

# Avg Monetary by segment
mon_data = seg_summary['Avg_Monetary'].reindex(order_seg, fill_value=0)
axes[1].bar(order_seg, mon_data.values, color=cols, edgecolor='black')
for i, v in enumerate(mon_data):
    axes[1].text(i, v + 200, f'Rs {v:,.0f}', ha='center', fontsize=9)
axes[1].set_title('Avg Lifetime Spend by Segment', fontsize=14)
axes[1].set_ylabel('Avg Revenue (Rs)')

plt.tight_layout()
plt.show()

---
## STEP 8: Product-Level Deep Dive

In [ ]:
# Top 10 products by revenue
top_products = df.groupby(['category','product']).agg(
    Revenue=('net_sales','sum'),
    Orders =('order_id', 'count'),
    Profit =('profit',   'sum')
).reset_index().sort_values('Revenue', ascending=False).head(10)

top_products['Label'] = top_products['category'] + ' > ' + top_products['product']

plt.figure(figsize=(13, 6))
bars = plt.barh(range(len(top_products)), top_products['Revenue'],
                color=plt.cm.tab10.colors[:10])
plt.yticks(range(len(top_products)), top_products['Label'], fontsize=11)
for i, (_, row) in enumerate(top_products.iterrows()):
    plt.text(row['Revenue'] + 10000, i, f'Rs {row["Revenue"]:,.0f}', va='center', fontsize=9)
plt.title('Top 10 Products by Revenue', fontsize=16)
plt.xlabel('Revenue (Rs)')
plt.tight_layout()
plt.show()

---
## STEP 9: Seasonality & Heatmap Analysis

In [ ]:
# Orders by Day of Week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_rev = df.groupby('day_of_week')['net_sales'].sum().reindex(day_order)

# Orders by Month
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
month_rev = df.groupby('month_name')['net_sales'].sum().reindex(month_order)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].bar(day_order, day_rev.values,
            color=['tomato' if d in ['Saturday','Sunday'] else 'steelblue' for d in day_order],
            edgecolor='black', alpha=0.85)
axes[0].set_title('Revenue by Day of Week', fontsize=14)
axes[0].set_ylabel('Revenue (Rs)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(range(12), month_rev.values, color='steelblue', edgecolor='black', alpha=0.85)
axes[1].set_xticks(range(12))
axes[1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1].set_title('Revenue by Month (Seasonality)', fontsize=14)
axes[1].set_ylabel('Revenue (Rs)')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Seasonality Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pivot heatmap: Category x Month
pivot_heat = df.pivot_table(
    values='net_sales', index='category',
    columns='month', aggfunc='sum'
)
pivot_heat.columns = ['J','F','M','A','M ','J ','J ','A ','S','O','N','D']

plt.figure(figsize=(15, 6))
sns.heatmap(pivot_heat / 1000, annot=True, fmt='.0f',
            cmap='YlOrRd', linewidths=0.4,
            cbar_kws={'label': 'Revenue (Rs thousands)'})
plt.title('Revenue Heatmap: Category vs Month (Rs 000s)', fontsize=15)
plt.xlabel('Month')
plt.tight_layout()
plt.show()

---
## STEP 10: Discount & Return Analysis

In [ ]:
# Impact of discount on profit
disc_impact = df.groupby('discount_pct').agg(
    Orders =('order_id', 'count'),
    Avg_Profit=('profit', 'mean'),
    Return_Rate=('is_returned', 'mean')
).round(2)
disc_impact['Return_Rate'] *= 100

print('Discount Impact Analysis:')
print(disc_impact)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(disc_impact.index, disc_impact['Avg_Profit'],
            color=['green' if v > 0 else 'red' for v in disc_impact['Avg_Profit']],
            edgecolor='black', alpha=0.85)
axes[0].axhline(0, color='black', lw=1)
axes[0].set_title('Avg Profit per Order by Discount %', fontsize=13)
axes[0].set_xlabel('Discount %')
axes[0].set_ylabel('Avg Profit (Rs)')
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(disc_impact.index, disc_impact['Return_Rate'],
             'ro-', lw=2, ms=8)
axes[1].set_title('Return Rate by Discount %', fontsize=13)
axes[1].set_xlabel('Discount %')
axes[1].set_ylabel('Return Rate %')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Discount Strategy Analysis', fontsize=16)
plt.tight_layout()
plt.show()

print('\nKey Insight: Does higher discount = more returns?')
corr, p = stats.pearsonr(disc_impact.index, disc_impact['Return_Rate'])
print(f'Correlation (Discount vs Return Rate): r = {corr:.3f}, p = {p:.3f}')

---
## STEP 11: Executive Dashboard (Complete)

In [ ]:
fig = plt.figure(figsize=(22, 16))
fig.suptitle('RETAIL ANALYTICS EXECUTIVE DASHBOARD  |  2022-2023',
             fontsize=22, fontweight='bold', y=0.98)

gs = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.38)

# ── Row 1 ──────────────────────────────────────────────────────
# 1a. Monthly Revenue trend
ax1 = fig.add_subplot(gs[0, :2])
ax1.bar(range(len(monthly)), monthly['Revenue'],
        color=['steelblue' if y==2022 else 'tomato' for y in monthly['year']],
        alpha=0.8, edgecolor='white')
ax1.plot(range(len(monthly)), monthly['revenue_ma3'], 'k-', lw=2.5, label='3M MA')
ax1.set_title('Monthly Revenue (Blue=2022, Red=2023)', fontsize=12)
ax1.set_xticks(range(0, len(monthly), 2))
ax1.set_xticklabels(labels[::2], rotation=45, fontsize=7)
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# 1b. Revenue by Category pie
ax2 = fig.add_subplot(gs[0, 2])
cat_rev_pie = df.groupby('category')['net_sales'].sum()
ax2.pie(cat_rev_pie, labels=cat_rev_pie.index, autopct='%1.0f%%',
        startangle=90, colors=plt.cm.Set2.colors)
ax2.set_title('Revenue by Category', fontsize=12)

# 1c. Revenue by Region
ax3 = fig.add_subplot(gs[0, 3])
reg_rev = df.groupby('region')['net_sales'].sum().sort_values(ascending=True)
ax3.barh(reg_rev.index, reg_rev.values, color=plt.cm.Set1.colors[:5])
ax3.set_title('Revenue by Region', fontsize=12)
ax3.set_xlabel('Revenue (Rs)')

# ── Row 2 ──────────────────────────────────────────────────────
# 2a. Customer segment distribution
ax4 = fig.add_subplot(gs[1, 0])
seg_c = rfm['Segment'].value_counts().reindex(order_seg, fill_value=0)
ax4.bar(order_seg, seg_c.values, color=cols, edgecolor='black', alpha=0.85)
ax4.set_title('Customer Segments', fontsize=12)
ax4.set_ylabel('Customers')
ax4.tick_params(axis='x', rotation=25)

# 2b. Top 5 Products
ax5 = fig.add_subplot(gs[1, 1])
top5 = top_products.head(5)
ax5.barh(top5['product'], top5['Revenue'],
         color=plt.cm.tab10.colors[:5])
ax5.set_title('Top 5 Products', fontsize=12)
ax5.set_xlabel('Revenue (Rs)')

# 2c. Day of Week Revenue
ax6 = fig.add_subplot(gs[1, 2])
day_colors = ['tomato' if d in ['Saturday','Sunday'] else 'steelblue' for d in day_order]
ax6.bar(['M','T','W','T','F','S','S'], day_rev.values,
        color=day_colors, edgecolor='black')
ax6.set_title('Revenue by Day (Red=Weekend)', fontsize=12)
ax6.set_ylabel('Revenue (Rs)')

# 2d. Discount vs Avg Profit
ax7 = fig.add_subplot(gs[1, 3])
ax7.bar(disc_impact.index, disc_impact['Avg_Profit'],
        color=['green' if v > 0 else 'red' for v in disc_impact['Avg_Profit']],
        edgecolor='black', alpha=0.85)
ax7.axhline(0, color='black', lw=1)
ax7.set_title('Avg Profit by Discount %', fontsize=12)
ax7.set_xlabel('Discount %')
ax7.set_ylabel('Avg Profit (Rs)')

# ── Row 3 ──────────────────────────────────────────────────────
# 3a. Category heatmap (revenue by month)
ax8 = fig.add_subplot(gs[2, :2])
pivot_dash = df.pivot_table(
    values='net_sales', index='category', columns='month', aggfunc='sum'
) / 1000
pivot_dash.columns = ['J','F','M','A','M ','J ','J ','A ','S','O','N','D']
sns.heatmap(pivot_dash, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.3, ax=ax8, cbar=False, annot_kws={'size': 8})
ax8.set_title('Monthly Revenue by Category (Rs 000s)', fontsize=12)
ax8.set_xlabel('')

# 3b. Return rate by category
ax9 = fig.add_subplot(gs[2, 2])
ret_cat = df.groupby('category')['is_returned'].mean() * 100
ret_cat = ret_cat.sort_values(ascending=True)
ax9.barh(ret_cat.index, ret_cat.values, color='tomato', alpha=0.85, edgecolor='black')
ax9.set_title('Return Rate by Category (%)', fontsize=12)
ax9.set_xlabel('Return %')

# 3c. Monthly profit trend
ax10 = fig.add_subplot(gs[2, 3])
prof_colors = ['green' if v > 0 else 'red' for v in monthly['Profit']]
ax10.bar(range(len(monthly)), monthly['Profit'], color=prof_colors, alpha=0.85)
ax10.axhline(0, color='black', lw=1)
ax10.set_title('Monthly Profit', fontsize=12)
ax10.set_xticks(range(0, len(monthly), 3))
ax10.set_xticklabels(labels[::3], rotation=45, fontsize=7)

plt.savefig('retail_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('Dashboard saved as retail_dashboard.png')

---
## STEP 12: Business Recommendations Report

In [ ]:
best_cat    = cat_perf['Revenue'].idxmax()
best_region = region_perf['Revenue'].idxmax()
best_month  = month_rev.idxmax()
worst_cat   = cat_perf['Profit_Margin%'].idxmin()
high_return = ret_cat.idxmax()

print('=' * 65)
print('  EXECUTIVE SUMMARY & RECOMMENDATIONS')
print('  Retail Analytics Report | Jan 2022 – Dec 2023')
print('=' * 65)

print(f'''
BUSINESS OVERVIEW
─────────────────
• Total Revenue  : Rs {df["net_sales"].sum():>12,.0f}
• Total Profit   : Rs {df["profit"].sum():>12,.0f}
• Profit Margin  : {df["profit"].sum()/df["net_sales"].sum()*100:.1f}%
• Orders Placed  : {len(df):>12,}
• Unique Customers: {df["customer_id"].nunique():>11,}
• Return Rate    : {df["is_returned"].mean()*100:.1f}%

KEY FINDINGS
────────────
1. BEST CATEGORY    : {best_cat} — highest revenue generator
2. BEST REGION      : {best_region} — strongest market
3. PEAK MONTH       : {best_month} — seasonal demand spike
4. LOW MARGIN       : {worst_cat} — needs pricing review
5. HIGH RETURNS     : {high_return} — quality/expectation mismatch
6. YoY GROWTH       : {(y2023/y2022-1)*100:.1f}% — business is growing

RECOMMENDATIONS
───────────────
1. INVEST  → Double down on {best_cat} — top revenue driver
2. EXPAND  → Grow presence in {best_region} through targeted campaigns
3. FESTIVE → Plan inventory 2 months before Nov-Dec for peak demand
4. PRICING → Review {worst_cat} pricing to improve profit margin
5. RETURNS → Investigate {high_return} returns — improve product quality or descriptions
6. LOYALTY → Convert 'At Risk' customers with re-engagement offers
7. DISCOUNT→ Avoid >20% discounts — they hurt profit without reducing returns
8. WEEKEND → Run promotions on Sat-Sun when order volume is highest
''')

print('=' * 65)
print('  Report prepared using Python Data Analytics')
print('  Tools: Pandas, NumPy, Matplotlib, Seaborn, SciPy')
print('=' * 65)

---
## What You Used in This Project

| Skill | Where Used |
|-------|------------|
| Pandas DataFrame | Creating & manipulating data |
| GroupBy + Aggregation | Category, region, customer analysis |
| Pivot Tables | Heatmap data |
| Time Series | Monthly trend, growth rate |
| Matplotlib | All bar/line/pie charts |
| Seaborn | Heatmaps, box plots |
| SciPy stats | Correlation p-values |
| Feature Engineering | RFM scores, segments, growth % |
| NumPy | Random data generation |
| Business Thinking | Turning data → decisions |

---
## This is a Job-Ready Analyst Portfolio Project!

You can:
- Put this on your resume: *"Built retail sales analytics dashboard using Python"*
- Upload to GitHub with a README
- Present to any company in an interview
- Adapt it for any industry (hospital, school, restaurant)

---
## Further Projects to Build

1. **E-commerce Churn Prediction** — who will stop buying?
2. **Inventory Forecasting** — how much stock to order?
3. **Price Elasticity Analysis** — how does price affect sales?
4. **Market Basket Analysis** — which products are bought together?
5. **Customer Lifetime Value** — how much is each customer worth?